In [1]:
import pandas as pd
import torch

In [2]:
import torch.nn as nn

In [4]:
data = pd.read_csv("DateFruit_Dataset.csv")

In [5]:
X = data.drop(columns="Class")
y = data["Class"]

from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42)

In [6]:
# Encode the output

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_encode_train = le.fit_transform(y_train)
y_encode_test = le.fit_transform(y_test)

In [7]:
# now standardize the data

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scale = scaler.fit_transform(X_train)
X_test_scale = scaler.fit_transform(X_test)

In [8]:
# now convert data to tensors

X_train_tensor = torch.tensor(X_train_scale,dtype=torch.float32)
y_train_tensor = torch.tensor(y_encode_train,dtype=torch.long)

X_test_tensor = torch.tensor(X_test_scale,dtype=torch.float32)
y_test_tensor = torch.tensor(y_encode_test,dtype=torch.long)

In [9]:
# Data Pipeline -> tensor dataset and data loader

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

train_dataset = TensorDataset(X_train_tensor,y_train_tensor)
test_dataset = TensorDataset(X_test_tensor,y_test_tensor)

In [10]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32)

In [11]:

class ANN(nn.Module):

    def __init__(self):

        super(ANN, self).__init__()

        self.model = nn.Sequential(

            # 1st hidden layer
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),

            # 2nd hidden layer
            nn.Linear(64, 64),
            nn.ReLU(),

            # output layer
            nn.Linear(64, 7)

        )

    # OUTSIDE __init__
    def forward(self, x):

        return self.model(x)

In [12]:
#loss and optimizers

import torch.optim as optim

model = ANN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

In [13]:
# Training the ANN model

epochs = 100
best_model = float("inf")

for i in range(epochs):
  model.train()
  loss_per = 0.0

  for xb,yb in train_loader:
    output = model(xb)

    loss = criterion(output,yb)
    loss_per += loss.item()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  avg_loss = loss_per/(len(train_loader))

# if(best_model>loss_per):
#   best_model = loss_per
#   torch.save(model.state_dict(),"best_classification_ANN_model")

  print("per epoch loss",i+1,":  ",avg_loss)

per epoch loss 1 :   1.7506054097955877
per epoch loss 2 :   1.16245037317276
per epoch loss 3 :   0.7532554742118175
per epoch loss 4 :   0.5941561365669424
per epoch loss 5 :   0.4675933017649434
per epoch loss 6 :   0.4402236965569583
per epoch loss 7 :   0.3803261775862087
per epoch loss 8 :   0.347353918985887
per epoch loss 9 :   0.29980115592479706
per epoch loss 10 :   0.2805464103479277
per epoch loss 11 :   0.27393164147030225
per epoch loss 12 :   0.30653514713048935
per epoch loss 13 :   0.2743499069051309
per epoch loss 14 :   0.23123543768782506
per epoch loss 15 :   0.21746257418205708
per epoch loss 16 :   0.2107987327670509
per epoch loss 17 :   0.197063202096615
per epoch loss 18 :   0.18878402351401746
per epoch loss 19 :   0.18745526823807845
per epoch loss 20 :   0.17802036668978294
per epoch loss 21 :   0.17062367021042685
per epoch loss 22 :   0.1715867268768224
per epoch loss 23 :   0.15844922318038615
per epoch loss 24 :   0.154443639584563
per epoch loss 25 : 

In [14]:
# Evaluation of the model

model.eval()

total=0
correct=0

with torch.no_grad():
  for xb,yb in test_loader:
    output = model(xb)

    _,predicted = torch.max(output,1)

    correct += (predicted==yb).sum().item()
    total += yb.size(0)

In [15]:
from sklearn.metrics import accuracy_score

print("Accuracy score is: ",correct/total)

Accuracy score is:  0.9288888888888889


# Solving the problem by using PCA with ANN

In [16]:
from sklearn.decomposition import PCA

pca = PCA(n_components=15,random_state=42)

X_train_pca = pca.fit_transform(X_train_scale)
X_test_pca = pca.transform(X_test_scale)

In [17]:
X_pca__train_tensor = torch.tensor(X_train_pca,dtype=torch.float32)
y_train_tensor = torch.tensor(y_encode_train,dtype=torch.long)

X_pca_test_tensor = torch.tensor(X_test_pca,dtype=torch.float32)
y_test_tensor = torch.tensor(y_encode_test,dtype=torch.long)

In [18]:
# tensor dataset and dataloader

train_pca_dataset = TensorDataset(X_pca__train_tensor,y_train_tensor)
test_pca_dataset = TensorDataset(X_pca_test_tensor,y_test_tensor)

In [19]:
train_pca_loader = DataLoader(train_pca_dataset,batch_size=32,shuffle=True)
test_pca_loader = DataLoader(test_pca_dataset,batch_size=32)

In [20]:


class ANN_PCA(nn.Module):

    def __init__(self):

        super(ANN_PCA, self).__init__()

        self.model = nn.Sequential(

            # 1st hidden layer
            nn.Linear(X_train_pca.shape[1], 64),
            nn.ReLU(),

            # 2nd hidden layer
            nn.Linear(64, 64),
            nn.ReLU(),

            # output layer
            nn.Linear(64, 7)

        )

    # OUTSIDE __init__
    def forward(self, x):

        return self.model(x)

In [21]:
#loss and optimizers

model_pca = ANN_PCA()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_pca.parameters(),lr=0.001)

In [22]:
# Training the ANN model on dataset(PCA)

epochs = 100
best_model = float("inf")

for i in range(epochs):
  model_pca.train()
  loss_per = 0.0

  for xb,yb in train_pca_loader:
    output = model_pca(xb)

    loss = criterion(output,yb)
    loss_per += loss.item()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  avg_loss = loss_per/(len(train_pca_loader))

# if(best_model>loss_per):
#   best_model = loss_per
#   torch.save(model.state_dict(),"best_classification_ANN_model")

  print("per epoch loss",i+1,":  ",avg_loss)

per epoch loss 1 :   1.652808820659464
per epoch loss 2 :   1.171750271184878
per epoch loss 3 :   0.8237082020125606
per epoch loss 4 :   0.6775806790048425
per epoch loss 5 :   0.5413417518138885
per epoch loss 6 :   0.42585963010787964
per epoch loss 7 :   0.39219140396876767
per epoch loss 8 :   0.3418459852162579
per epoch loss 9 :   0.31499447351829574
per epoch loss 10 :   0.284110309725458
per epoch loss 11 :   0.2628501078685407
per epoch loss 12 :   0.30729499933394516
per epoch loss 13 :   0.24124275105846624
per epoch loss 14 :   0.22896346619183366
per epoch loss 15 :   0.20805158419005404
per epoch loss 16 :   0.19813179658508903
per epoch loss 17 :   0.19132597338069568
per epoch loss 18 :   0.20112655718218198
per epoch loss 19 :   0.17959119274217467
per epoch loss 20 :   0.16788370071762157
per epoch loss 21 :   0.16043305426666682
per epoch loss 22 :   0.15392223064025695
per epoch loss 23 :   0.1496379982695957
per epoch loss 24 :   0.14536984224634414
per epoch los

In [23]:
# Evaluation

model_pca.eval()

total =0
correct=0

for xb,yb in test_pca_loader:
  output = model_pca(xb)

  _,predict = torch.max(output,1)

  correct += (yb==predict).sum().item()
  total += yb.size(0)


print("Corrected outputs: ",correct)
print("Total number of inputs: ",total)

Corrected outputs:  207
Total number of inputs:  225


In [24]:
# check for Acuuracy

print("Accuracy of the model by doing PCA: ",correct/total)

Accuracy of the model by doing PCA:  0.92


### ANN without PCA performed little better than ANN with PCA, achieving 92.9% accuracy compared to 92% after applying PCA.